# Bayes-Kalman Filters, Dynamic Programming and Hidden Markov Models

Prepared for Fall 2026. Reference: the notes for Lecture 9; page numbers refer to them. If you find a bug, please email the TAs or tell us at office hours.

**Before you start.** Everything is simulated with numpy under fixed seeds, so you get our numbers. Every figure and number a question asks for must appear in your PDF: to vary a parameter, copy the cell below the question (or loop) rather than re-running it above.

---

**The Bayes-Kalman filter (pp. 12–30)** estimates the state $x_t$ from the observations $Y_t = \{y_t, \ldots, y_1\}$: a prediction with the motion model, $P(x_{t+1} \mid Y_t) = \int dx_t \, P(x_{t+1} \mid x_t) \, P(x_t \mid Y_t)$, then a correction by Bayes' rule with the observation model $P(y_{t+1} \mid x_{t+1})$. With Gaussian noise ($\sigma_m$ on the observations, $\sigma_p$ on the motion, mean step $\mu$) and a Gaussian prior the estimate is $N(\mu_t, \sigma_t^2)$, the prediction $N(\mu + \mu_t, \, \sigma_p^2 + \sigma_t^2)$, and the correction a weighted average of the two, each weighted by the variance of the other, whose precisions add:

$$\mu_{t+1} = \frac{\sigma_m^2}{\sigma_m^2 + \sigma_p^2 + \sigma_t^2} \, (\mu + \mu_t) + K_{t+1} \, y_{t+1}, \qquad K_{t+1} = \frac{\sigma_p^2 + \sigma_t^2}{\sigma_m^2 + \sigma_p^2 + \sigma_t^2}, \qquad \frac{1}{\sigma_{t+1}^2} = \frac{1}{\sigma_p^2 + \sigma_t^2} + \frac{1}{\sigma_m^2};$$

$K_{t+1}$, the observation's weight, is the Kalman gain. The variance recursion contains no observation, so the $\sigma_t^2$ are known in advance and converge to the fixed point

$$\sigma^2 = \tfrac{1}{2} \Big( -\sigma_p^2 + \sqrt{\sigma_p^4 + 4 \sigma_m^2 \sigma_p^2} \Big), \qquad K = \frac{\sigma_p^2 + \sigma^2}{\sigma_m^2 + \sigma_p^2 + \sigma^2}.$$

## Utilities

In [ ]:
from __future__ import annotations
import itertools

import numpy as np
import numpy.typing as npt
import matplotlib.pyplot as plt

%matplotlib inline
np.set_printoptions(precision=3, suppress=True)

COLOUR = {"truth": "tab:red", "observation": "tab:blue", "prediction": "tab:orange", "estimate": "teal", "greedy": "olive"}


def fmt(values, digits: int = 2) -> str:
    """Format a sequence of numbers with a fixed number of decimals, separated by spaces."""
    return " ".join(f"{v:.{digits}f}" for v in np.atleast_1d(values))


def rms(a, b) -> float:
    """Root-mean-square difference between two arrays of the same length."""
    return float(np.sqrt(np.mean((np.asarray(a, dtype=float) - np.asarray(b, dtype=float)) ** 2)))


# ------------------------------------------------------------------ 1. The Bayes-Kalman filter (Gaussian case)

def simulate_track(n_frames: int = 24, mu: float = 0.25, sigma_p: float = 0.3, sigma_m: float = 1.0,
                   mu0: float = 0.0, sigma0: float = 2.0, x1: float | None = None, seed: int = 0):
    """Simulate the Gaussian model of the notes (p. 22): x_1 ~ N(mu0, sigma0^2) (or x_1 = x1 if given),
    x_{t+1} = x_t + mu + N(0, sigma_p^2), y_t = x_t + N(0, sigma_m^2).
    Returns the true positions x and the observations y, arrays of length n_frames (index 0 is frame t = 1)."""
    rng = np.random.default_rng(seed)
    x = np.empty(n_frames)
    x[0] = rng.normal(mu0, sigma0) if x1 is None else x1
    for t in range(1, n_frames):
        x[t] = x[t - 1] + mu + rng.normal(0.0, sigma_p)
    y = x + rng.normal(0.0, sigma_m, size=n_frames)
    return x, y


def kalman_filter(y, mu: float, sigma_p: float, sigma_m: float, mu0: float, sigma0: float, observed=None) -> dict:
    """The Bayes-Kalman filter for the Gaussian model (pp. 22–25): one prediction and one correction per frame.

    y        : the observations y_1, ..., y_T
    observed : optional boolean array of the same length; where it is False the observation of that frame is
               missing, only the prediction is applied, the gain is 0 and the estimate is the prediction
    Returns a dict of arrays indexed by frame (index 0 is t = 1):
      "pred_mean", "pred_var" : the prediction N(mu + mu_{t-1}, sigma_p^2 + sigma_{t-1}^2) (at t = 1: the prior),
      "mean", "var"           : the estimate N(mu_t, sigma_t^2) after the correction,
      "gain"                  : the Kalman gain K_t.
    """
    y = np.asarray(y, dtype=float)
    T = len(y)
    observed = np.ones(T, dtype=bool) if observed is None else np.asarray(observed, dtype=bool)
    out = {key: np.empty(T) for key in ("pred_mean", "pred_var", "mean", "var", "gain")}
    pred_mean, pred_var = mu0, sigma0**2                        # at t = 1 the "prediction" is the prior N(mu0, sigma0^2)
    for t in range(T):
        if t > 0:
            pred_mean = mu + out["mean"][t - 1]                   # prediction: mean step added to the last estimate
            pred_var = sigma_p**2 + out["var"][t - 1]             #             motion variance added to its variance
        if observed[t]:
            total = sigma_m**2 + pred_var
            if total > 0:
                K = pred_var / total                              # the gain pred_var / (sigma_m^2 + pred_var): the notes' K_{t+1} = (sigma_p^2 + sigma_t^2) / (sigma_m^2 + sigma_p^2 + sigma_t^2)
                mean = pred_mean + K * (y[t] - pred_mean)         # correction: prediction + K_t * innovation
                var = sigma_m**2 * pred_var / total               #             sigma_t^2 = sigma_m^2 v / (sigma_m^2 + v)
            else:                                                 # both variances are 0 (0/0): the observation is exact
                K, mean, var = 1.0, y[t], 0.0
        else:                                                     # no observation: keep the prediction
            K, mean, var = 0.0, pred_mean, pred_var
        out["pred_mean"][t], out["pred_var"][t] = pred_mean, pred_var
        out["mean"][t], out["var"][t], out["gain"][t] = mean, var, K
    return out


def steady_state(sigma_p: float, sigma_m: float):
    """Fixed point of the variance recursion (p. 30), sigma^2 = (-sigma_p^2 + sqrt(sigma_p^4 + 4 sigma_m^2 sigma_p^2)) / 2,
    and the gain there, K = (sigma_p^2 + sigma^2) / (sigma_m^2 + sigma_p^2 + sigma^2). Returns (sigma, K)."""
    var = 0.5 * (-sigma_p**2 + np.sqrt(sigma_p**4 + 4 * sigma_m**2 * sigma_p**2))
    total = sigma_m**2 + sigma_p**2 + var
    K = (sigma_p**2 + var) / total if total > 0 else 1.0
    return float(np.sqrt(var)), float(K)


def batch_posterior(y, sigma_m: float, mu0: float, sigma0: float):
    """Posterior of a static object after t = 1, ..., T measurements, computed in one batch (p. 32):
    1/sigma_t^2 = 1/sigma_0^2 + t/sigma_m^2,  mu_t = sigma_t^2 (mu_0/sigma_0^2 + sum_{s<=t} y_s / sigma_m^2).
    Returns the means mu_t and the variances sigma_t^2 for every t."""
    y = np.asarray(y, dtype=float)
    t = np.arange(1, len(y) + 1)
    var = 1.0 / (1.0 / sigma0**2 + t / sigma_m**2)
    mean = var * (mu0 / sigma0**2 + np.cumsum(y) / sigma_m**2)
    return mean, var


def plot_track(x, y, est: dict, title: str, observed=None, show_prediction: bool = False, ax=None):
    """Truth (red line), observations (blue dots), estimate mu_t (teal line) with the band mu_t ± sigma_t, and
    optionally the prediction (orange dashed line with its band, drawn from t = 2 on; at t = 1 it is the prior).
    Missing observations (observed = False) are not drawn."""
    x, y = np.asarray(x), np.asarray(y)
    t = np.arange(1, len(x) + 1)
    observed = np.ones(len(x), dtype=bool) if observed is None else np.asarray(observed, dtype=bool)
    own_figure = ax is None
    if own_figure:
        fig, ax = plt.subplots(figsize=(7.5, 3.4))
    sd = np.sqrt(est["var"])
    ax.fill_between(t, est["mean"] - sd, est["mean"] + sd, color=COLOUR["estimate"], alpha=0.2, label=r"$\mu_t \pm \sigma_t$")
    if show_prediction:
        psd = np.sqrt(est["pred_var"][1:])
        ax.plot(t[1:], est["pred_mean"][1:], color=COLOUR["prediction"], ls="--", lw=1.3, label=r"prediction $\mu + \mu_{t-1}$")
        ax.fill_between(t[1:], est["pred_mean"][1:] - psd, est["pred_mean"][1:] + psd, color=COLOUR["prediction"], alpha=0.15)
    ax.plot(t, x, color=COLOUR["truth"], lw=1.5, label=r"true position $x_t$")
    ax.plot(t[observed], y[observed], "o", color=COLOUR["observation"], ms=4, label=r"observations $y_t$")
    ax.plot(t, est["mean"], color=COLOUR["estimate"], lw=2, label=r"estimate $\mu_t$")
    ax.set_xlabel("frame $t$")
    ax.set_ylabel("position")
    ax.set_title(title)
    ax.legend(fontsize=8, ncol=2)
    if own_figure:
        plt.show()
    return ax


def plot_sigma_and_gain(est: dict, sigma_inf: float | None = None, K_inf: float | None = None, title: str = ""):
    """sigma_t and the gain K_t against the frame, with the fixed-point values as dashed lines if given."""
    t = np.arange(1, len(est["var"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(9, 2.8))
    axes[0].plot(t, np.sqrt(est["var"]), "o-", ms=3, color=COLOUR["estimate"], label=r"$\sigma_t$")
    axes[1].plot(t, est["gain"], "o-", ms=3, color=COLOUR["estimate"], label=r"$K_t$")
    if sigma_inf is not None:
        axes[0].axhline(sigma_inf, ls="--", color="k", lw=1, label=f"fixed point {sigma_inf:.2f}")
    if K_inf is not None:
        axes[1].axhline(K_inf, ls="--", color="k", lw=1, label=f"fixed point {K_inf:.2f}")
    for ax, name in zip(axes, (r"posterior standard deviation $\sigma_t$", r"Kalman gain $K_t$")):
        ax.set_xlabel("frame $t$")
        ax.set_ylabel(name.split()[-1])
        ax.set_title(f"{name} {title}".strip())
        ax.set_ylim(bottom=0)
        ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()


# ------------------------------------------------------------------ 2. Dynamic programming on a chain

def chain_energy(labels, unary, pairwise) -> float:
    """E(x) = sum_m psi_m(x_m) + sum_m phi_{m,m+1}(x_m, x_{m+1}) for one labelling (unary (N, k); pairwise (k, k) or (N-1, k, k))."""
    labels = np.asarray(labels, dtype=int)
    unary, pairwise = np.asarray(unary, dtype=float), np.asarray(pairwise, dtype=float)
    N = len(labels)
    if labels.min() < 0 or labels.max() >= unary.shape[1]:
        raise ValueError(f"labels must be values 0..{unary.shape[1] - 1} (the tables have k = {unary.shape[1]} values per "
                         f"variable); got {labels.min()}..{labels.max()}")
    if pairwise.ndim == 2:
        pairwise = np.broadcast_to(pairwise, (N - 1, *pairwise.shape))
    return float(unary[np.arange(N), labels].sum() + pairwise[np.arange(N - 1), labels[:-1], labels[1:]].sum())


def dp_chain(unary, pairwise):
    """Dynamic programming on a chain of N variables with k values each (pp. 64–70).

    unary   : (N, k) array, unary[m, a] = psi_m(a), the cost of value a at variable m (0 if there is none)
    pairwise: (k, k) array, pairwise[a, b] = phi(a, b), the cost of the pair x_m = a, x_{m+1} = b, the same for every
              pair, or an (N-1, k, k) array with one table phi_{m,m+1} per pair
    Forward pass: h_1(x_1) = psi_1(x_1) and
        h_m(x_m) = psi_m(x_m) + min_{x_{m-1}} {h_{m-1}(x_{m-1}) + phi_{m-1,m}(x_{m-1}, x_m)},
    storing the minimiser, the best way into x_m (ties are broken by the first index).
    Backward pass: x_N = argmin h_N(x_N), then x_{m-1} = the stored minimiser of x_m, down to x_1.
    Returns the labels x_1..x_N (N,), the minimum cost, the table h (N, k) and the stored minimisers (N, k), the best
    way into each value."""
    unary, pairwise = np.asarray(unary, dtype=float), np.asarray(pairwise, dtype=float)
    N, k = unary.shape
    if pairwise.ndim == 2:
        pairwise = np.broadcast_to(pairwise, (N - 1, k, k))
    h = np.empty((N, k))
    back = np.zeros((N, k), dtype=int)
    h[0] = unary[0]
    for m in range(1, N):
        candidates = h[m - 1][:, None] + pairwise[m - 1]      # candidates[a, b] = h_{m-1}(a) + phi_{m-1,m}(a, b)
        back[m] = np.argmin(candidates, axis=0)              # the best way into each value b of x_m
        h[m] = candidates.min(axis=0) + unary[m]
    labels = np.empty(N, dtype=int)
    labels[-1] = int(np.argmin(h[-1]))
    for m in range(N - 1, 0, -1):
        labels[m - 1] = back[m, labels[m]]
    return labels, float(h[-1, labels[-1]]), h, back


def brute_force_chain(unary, pairwise):
    """Try all k^N configurations of the chain (small N only). Returns the best labels and the minimum cost."""
    unary = np.asarray(unary, dtype=float)
    N, k = unary.shape
    best, best_cost = None, np.inf
    for config in itertools.product(range(k), repeat=N):
        cost = chain_energy(config, unary, pairwise)
        if cost < best_cost:
            best, best_cost = np.array(config), cost
    return best, best_cost


def make_step_signal(noise: float = 0.5, seed: int = 0):
    """A piecewise-constant signal with the k = 5 grey levels 0, ..., 4 and its noisy observation z_i = x_i + N(0, noise^2).
    Returns the true levels (integers) and z."""
    segments = [(1, 16), (2, 6), (3, 12), (4, 10), (3, 3), (4, 13), (0, 20)]     # (grey level, length); N = 80
    truth = np.concatenate([np.full(length, level) for level, length in segments])
    rng = np.random.default_rng(seed)
    return truth, truth + rng.normal(0.0, noise, size=len(truth))


def potts_costs(z, k: int, w: float):
    """Unary psi_i(v) = (z_i - v)^2 for the grey levels v = 0, ..., k-1 (array (N, k)) and the Potts pair cost
    phi(a, b) = w [a != b] (array (k, k)), with w the smoothness weight."""
    levels = np.arange(k)
    unary = (np.asarray(z, dtype=float)[:, None] - levels[None, :]) ** 2
    pairwise = w * (levels[:, None] != levels[None, :]).astype(float)
    return unary, pairwise


def plot_labellings(z, truth, labellings: dict, k: int):
    """One row per labelling: the observations z (blue dots), the true levels (red steps) and the labelling (teal steps);
    below them the same labellings as an image, one row each, grey levels 0 (black) to k-1 (white)."""
    n = len(labellings)
    fig, axes = plt.subplots(n + 1, 1, figsize=(8, 1.7 * n + 1.6), gridspec_kw={"height_ratios": [1] * n + [0.9]}, sharex=True)
    i = np.arange(1, len(z) + 1)
    for ax, (name, labels) in zip(axes[:-1], labellings.items()):
        ax.plot(i, z, "o", color=COLOUR["observation"], ms=2.5, label="$z_i$")
        ax.step(i, truth, where="mid", color=COLOUR["truth"], lw=1.2, label="true level")
        ax.step(i, labels, where="mid", color=COLOUR["estimate"], lw=2, label="DP labelling")
        ax.set_ylabel("grey level")
        ax.set_title(name, fontsize=10)
        ax.set_yticks(range(k))
    axes[0].legend(fontsize=8, ncol=3, loc="upper left")
    image = np.array([labels for labels in labellings.values()])
    axes[-1].imshow(image, cmap="gray", vmin=0, vmax=k - 1, aspect="auto", interpolation="nearest",
                    extent=(0.5, len(z) + 0.5, n - 0.5, -0.5))
    axes[-1].set_yticks(range(n))
    axes[-1].set_yticklabels(list(labellings.keys()), fontsize=8)
    axes[-1].set_title(f"the labellings as images (black = 0, white = {k - 1})", fontsize=10)
    axes[-1].set_xlabel("position $i$")
    fig.tight_layout()
    plt.show()


def plot_dp_table(h, labels, title: str):
    """The forward table h_m(x_m) as an image (rows = values, columns = variables) with the DP path in teal."""
    N, k = h.shape
    fig, ax = plt.subplots(figsize=(8, 2.6))
    im = ax.imshow(h.T, cmap="magma", aspect="auto", origin="lower", interpolation="nearest",
                   extent=(0.5, N + 0.5, -0.5, k - 0.5))
    ax.plot(np.arange(1, N + 1), labels, "o-", color=COLOUR["estimate"], ms=4, lw=1, mec="w", mew=0.6, label="DP labelling")
    ax.set_xlabel("variable $m$ (position $i$)")
    ax.set_ylabel("value $x_m$ (grey level)")
    ax.set_yticks(range(k))
    ax.set_title(title)
    ax.legend(fontsize=8, loc="upper left")
    fig.colorbar(im, ax=ax, label="$h_m(x_m)$")
    plt.show()


# ------------------------------------------------------------------ 3. Hidden Markov models

def forward(A, B, pi, O):
    """Forward variables alpha_t(i) = P(O_1, ..., O_t, q_t = s_i | lambda) (pp. 99–100).
    A (N, N) transition probabilities a_ij, B (N, M) observation probabilities b_j(m), pi (N,) initial probabilities,
    O (T,) the observed symbols coded 0, ..., M-1. Returns alpha (T, N) and P(O | lambda) = sum_i alpha_T(i)."""
    A, B, pi, O = np.asarray(A, dtype=float), np.asarray(B, dtype=float), np.asarray(pi, dtype=float), np.asarray(O, dtype=int)
    T, N = len(O), len(pi)
    alpha = np.empty((T, N))
    alpha[0] = pi * B[:, O[0]]                     # initialisation: alpha_1(i) = pi_i b_i(O_1); index 0 is t = 1
    for t in range(1, T):
        summed = alpha[t - 1] @ A                  # sum_i alpha[t-1](i) a_ij for every j (the notes' sum_i alpha_t(i) a_ij)
        alpha[t] = summed * B[:, O[t]]             # times b_j(O[t]), the observation of this step (the notes' b_j(O_{t+1}))
    return alpha, float(alpha[-1].sum())


def backward(A, B, pi, O):
    """Backward variables beta_t(i) = P(O_{t+1}, ..., O_T | q_t = s_i, lambda) (p. 104):
    beta_T(i) = 1, beta_t(i) = sum_j a_ij b_j(O_{t+1}) beta_{t+1}(j). Returns beta (T, N)."""
    A, B, O = np.asarray(A, dtype=float), np.asarray(B, dtype=float), np.asarray(O, dtype=int)
    T, N = len(O), A.shape[0]
    beta = np.empty((T, N))
    beta[-1] = 1.0
    for t in range(T - 2, -1, -1):
        beta[t] = A @ (B[:, O[t + 1]] * beta[t + 1])
    return beta


def state_posteriors(A, B, pi, O):
    """gamma_t(i) = P(q_t = s_i | O, lambda) = alpha_t(i) beta_t(i) / sum_j alpha_t(j) beta_t(j) (p. 105).
    Returns gamma (T, N) and P(O | lambda)."""
    alpha, p_O = forward(A, B, pi, O)
    if not p_O > 0:
        raise ValueError("P(O | lambda) = 0: the observations are impossible under (A, B, pi) (or the forward probabilities "
                         "underflowed for a long sequence), so gamma_t(i) = 0/0 is undefined")
    beta = backward(A, B, pi, O)
    joint = alpha * beta
    return joint / joint.sum(axis=1, keepdims=True), p_O


def viterbi(A, B, pi, O):
    """The Viterbi algorithm (pp. 110–111), ties broken by the first index.
    delta_1(i) = pi_i b_i(O_1), psi_1(i) = 0; delta_t(j) = max_i delta_{t-1}(i) a_ij b_j(O_t), psi_t(j) = argmax_i delta_{t-1}(i) a_ij;
    p* = max_i delta_T(i), q*_T = argmax_i delta_T(i); q*_t = psi_{t+1}(q*_{t+1}).
    Returns the path q*_1..q*_T (state indices 0, ..., N-1), p* = P(Q*, O | lambda), delta (T, N) and psi (T, N)."""
    A, B, pi, O = np.asarray(A, dtype=float), np.asarray(B, dtype=float), np.asarray(pi, dtype=float), np.asarray(O, dtype=int)
    T, N = len(O), len(pi)
    delta, psi = np.empty((T, N)), np.zeros((T, N), dtype=int)
    delta[0] = pi * B[:, O[0]]
    for t in range(1, T):
        candidates = delta[t - 1][:, None] * A           # candidates[i, j] = delta_{t-1}(i) a_ij
        psi[t] = np.argmax(candidates, axis=0)
        delta[t] = candidates.max(axis=0) * B[:, O[t]]
    path = np.empty(T, dtype=int)
    path[-1] = int(np.argmax(delta[-1]))
    for t in range(T - 2, -1, -1):
        path[t] = psi[t + 1, path[t + 1]]
    return path, float(delta[-1, path[-1]]), delta, psi


def sequence_probability(A, B, pi, O, Q) -> float:
    """P(Q, O | lambda) = pi_{q_1} b_{q_1}(O_1) prod_{t=2}^T a_{q_{t-1} q_t} b_{q_t}(O_t) for one state sequence Q (p. 98)."""
    A, B, pi = np.asarray(A, dtype=float), np.asarray(B, dtype=float), np.asarray(pi, dtype=float)
    O, Q = np.asarray(O, dtype=int), np.asarray(Q, dtype=int)
    p = pi[Q[0]] * B[Q[0], O[0]]
    for t in range(1, len(O)):
        p *= A[Q[t - 1], Q[t]] * B[Q[t], O[t]]
    return float(p)


def brute_force_hmm(A, B, pi, O):
    """P(Q, O | lambda) for every one of the N^T state sequences (small T only).
    Returns the best sequence, its probability, and the sum over all sequences (which is P(O | lambda))."""
    N, T = len(pi), len(O)
    best, best_p, total = None, -1.0, 0.0
    for Q in itertools.product(range(N), repeat=T):
        p = sequence_probability(A, B, pi, O, Q)
        total += p
        if p > best_p:
            best, best_p = np.array(Q), p
    return best, best_p, total


def simulate_markov_chain(A, pi, T: int, seed: int = 0):
    """Draw one state sequence q_1, ..., q_T (coded 0, ..., N-1) from the Markov model (A, pi)."""
    A, pi = np.asarray(A, dtype=float), np.asarray(pi, dtype=float)
    rng = np.random.default_rng(seed)
    states = np.empty(T, dtype=int)
    states[0] = rng.choice(len(pi), p=pi)
    for t in range(1, T):
        states[t] = rng.choice(len(pi), p=A[states[t - 1]])
    return states


def simulate_hmm(A, B, pi, T: int, n_sequences: int = 1, seed: int = 0):
    """Draw n_sequences state sequences of length T from (A, pi) and one observation per state from B.
    Returns the states (n_sequences, T) and the observations (n_sequences, T), both coded from 0."""
    B = np.asarray(B, dtype=float)
    rng = np.random.default_rng(seed)
    states = np.array([simulate_markov_chain(A, pi, T, seed=int(s)) for s in rng.integers(0, 2**31, size=n_sequences)])
    observations = np.array([[rng.choice(B.shape[1], p=B[q]) for q in row] for row in states])
    return states, observations


def plot_grid(gamma, A, O, ball_colours, greedy_path=None, viterbi_path=None, title: str = ""):
    """The grid of states against time: rows are the states, columns the times; each node is shaded by gamma_t(i) (white 0 to black 1,
    value written inside), grey lines are the allowed transitions (a_ij > 0), the greedy choice is joined in olive
    and the Viterbi path in teal; the observed balls are drawn below in their colours."""
    gamma, A = np.asarray(gamma), np.asarray(A)
    T, N = gamma.shape
    tt = np.arange(1, T + 1)
    fig, ax = plt.subplots(figsize=(1.7 * T + 3, 3.6))
    for t in range(T - 1):
        for i in range(N):
            for j in range(N):
                if A[i, j] > 0:
                    ax.plot([tt[t], tt[t + 1]], [i, j], color="0.8", lw=1, zorder=1)
    if greedy_path is not None:
        ax.plot(tt, greedy_path, color=COLOUR["greedy"], lw=4, alpha=0.8, label=r"greedy: $\arg\max_i \gamma_t(i)$", zorder=2)
    if viterbi_path is not None:
        ax.plot(tt, viterbi_path, color=COLOUR["estimate"], lw=2, ls="--", label="Viterbi path", zorder=3)
    sc = ax.scatter(np.repeat(tt, N), np.tile(np.arange(N), T), c=gamma.ravel(), cmap="Greys", vmin=0, vmax=1,
                    s=650, edgecolors="k", zorder=4)
    for t in range(T):
        for i in range(N):
            ax.text(tt[t], i, f"{gamma[t, i]:.2f}", ha="center", va="center", fontsize=8,
                    color="w" if gamma[t, i] > 0.5 else "k", zorder=5)
    ax.scatter(tt, np.full(T, N), c=[ball_colours[o] for o in O], s=130, edgecolors="k", zorder=4)
    ax.set_yticks(list(range(N)) + [N])
    ax.set_yticklabels([f"$s_{i + 1}$" for i in range(N)] + ["$O_t$"])
    ax.set_ylim(N + 0.6, -0.6)
    ax.set_xticks(tt)
    ax.set_xlim(0.4, T + 0.6)
    ax.set_xlabel("time $t$")
    ax.set_ylabel("state")
    ax.set_title(title)
    fig.colorbar(sc, ax=ax, label=r"$\gamma_t(i)$")
    ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.22, 1.0))
    plt.show()


def show_matrices(matrices, titles):
    """Transition matrices as grids: rows = state moved from, columns = state moved to, white 0 to black 1."""
    fig, axes = plt.subplots(1, len(matrices), figsize=(3.1 * len(matrices), 2.9))
    for ax, M, title in zip(np.atleast_1d(axes), matrices, titles):
        M = np.asarray(M)
        ax.imshow(M, cmap="Greys", vmin=0, vmax=1)
        for (i, j), v in np.ndenumerate(M):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", color="w" if v > 0.5 else "k")
        ax.set_xticks(range(M.shape[1]))
        ax.set_yticks(range(M.shape[0]))
        ax.set_xticklabels([f"$s_{j + 1}$" for j in range(M.shape[1])])
        ax.set_yticklabels([f"$s_{i + 1}$" for i in range(M.shape[0])])
        ax.set_xlabel("to state $j$")
        ax.set_ylabel("from state $i$")
        ax.set_title(title)
    fig.tight_layout()
    plt.show()


# ------------------------------------------------------------------ 4. Learning

def transition_counts(states, n_states: int):
    """Counts n_ij of the transitions s_i -> s_j in one observed state sequence (states coded 0, ..., n_states-1) and
    the maximum-likelihood estimate a_ij = n_ij / sum_j n_ij (pp. 84–85), for any number of states.
    A state that never has a successor in the sequence gives no information about its row; that row is set to
    the uniform distribution 1/n_states. Returns (counts, A_hat)."""
    states = np.asarray(states, dtype=int)
    counts = np.zeros((n_states, n_states))
    np.add.at(counts, (states[:-1], states[1:]), 1.0)
    row_totals = counts.sum(axis=1, keepdims=True)
    A_hat = np.where(row_totals > 0, counts / np.maximum(row_totals, 1.0), 1.0 / n_states)
    return counts, A_hat


def _forward_backward_scaled(A, B, pi, O, floor: float = 1e-300):
    """The forward and backward recursions with alpha_t and beta_t rescaled at every t (so that long sequences
    do not underflow). The posteriors gamma_t(i) and zeta_t(i, j) are unchanged by the rescaling, and the scale
    factors give log P(O | lambda). Returns gamma (T, N), zeta (T-1, N, N), log P(O | lambda)."""
    T, N = len(O), len(pi)
    alpha, scale = np.empty((T, N)), np.empty(T)
    a = pi * B[:, O[0]]
    scale[0] = max(a.sum(), floor)
    alpha[0] = a / scale[0]
    for t in range(1, T):
        a = (alpha[t - 1] @ A) * B[:, O[t]]
        scale[t] = max(a.sum(), floor)
        alpha[t] = a / scale[t]
    beta = np.empty((T, N))
    beta[-1] = 1.0
    for t in range(T - 2, -1, -1):
        beta[t] = (A @ (B[:, O[t + 1]] * beta[t + 1])) / scale[t + 1]
    gamma = alpha * beta
    gamma /= np.maximum(gamma.sum(axis=1, keepdims=True), floor)
    zeta = alpha[:-1, :, None] * A[None, :, :] * (B[:, O[1:]].T * beta[1:])[:, None, :]
    zeta /= np.maximum(zeta.sum(axis=(1, 2), keepdims=True), floor)
    return gamma, zeta, float(np.log(scale).sum())


def baum_welch(sequences, n_states: int, n_symbols: int, n_iter: int = 60, seed: int = 0):
    """The Baum-Welch algorithm, EM for an HMM (pp. 116–121), on K observation sequences, from a random start.

    E-step: gamma_t(i) and zeta_t(i, j) of every sequence by the forward-backward recursions.
    M-step: pi_i = (1/K) sum_k gamma^k_1(i),  a_ij = sum_k sum_t zeta^k_t(i, j) / sum_k sum_t gamma^k_t(i),
            b_j(m) = sum_k sum_t gamma^k_t(j) I(O^k_t = v_m) / sum_k sum_t gamma^k_t(j)   (p. 120).
    Returns A, B, pi and the history of log P(X | lambda) = sum_k log P(O^k | lambda): n_iter + 1 values, the
    first for the random start and one after every M-step."""
    rng = np.random.default_rng(seed)
    sequences = [np.asarray(O, dtype=int) for O in sequences]
    A = rng.random((n_states, n_states)) + 0.1
    B = rng.random((n_states, n_symbols)) + 0.1
    A, B = A / A.sum(axis=1, keepdims=True), B / B.sum(axis=1, keepdims=True)
    pi = np.full(n_states, 1.0 / n_states)
    history = []
    for iteration in range(n_iter + 1):
        pi_sum, zeta_sum, gamma_from, emit_sum, gamma_all = (np.zeros(n_states), np.zeros((n_states, n_states)),
                                                             np.zeros(n_states), np.zeros((n_states, n_symbols)), np.zeros(n_states))
        log_lik = 0.0
        for O in sequences:                                   # E-step
            gamma, zeta, ll = _forward_backward_scaled(A, B, pi, O)
            log_lik += ll
            pi_sum += gamma[0]
            zeta_sum += zeta.sum(axis=0)
            gamma_from += gamma[:-1].sum(axis=0)
            np.add.at(emit_sum.T, O, gamma)                    # emit_sum[j, m] += gamma_t(j) for every t with O_t = v_m
            gamma_all += gamma.sum(axis=0)
        history.append(log_lik)
        if iteration == n_iter:                               # the last E-step only evaluates the final parameters
            break
        pi = pi_sum / len(sequences)                          # M-step
        A = np.where(gamma_from[:, None] > 0, zeta_sum / np.maximum(gamma_from, 1e-300)[:, None], 1.0 / n_states)
        B = np.where(gamma_all[:, None] > 0, emit_sum / np.maximum(gamma_all, 1e-300)[:, None], 1.0 / n_symbols)
    return A, B, pi, np.array(history)


def match_states(B_hat):
    """Match the learnt states to the urns by their most likely colour (p. 122): the largest entry of B_hat pairs a
    state with a colour (= an urn), both are removed, and so on, so that every urn gets exactly one state.
    Returns the permutation perm with perm[urn] = learnt state; A_hat[perm][:, perm] and B_hat[perm] list the
    learnt states in the order of the urns."""
    table = np.array(B_hat, dtype=float)
    n_states, n_colours = table.shape
    if n_states != n_colours:
        raise ValueError(f"match_states pairs every urn with exactly one learnt state, so Baum-Welch must be run with "
                         f"n_states equal to the number of colours ({n_colours}); got {n_states}")
    n = n_states
    perm = np.full(n, -1, dtype=int)
    for _ in range(n):
        state, colour = np.unravel_index(np.argmax(table), table.shape)
        perm[colour] = state
        table[state, :] = -1.0
        table[:, colour] = -1.0
    return perm

## Tracking a moving target

The example of the notes (p. 28): $\mu = 0.25$, $\sigma_p = 0.3$, $\sigma_m = 1$, prior $N(0, 2^2)$, 24 frames. The cell runs the filter and compares the settled $\sigma_t$ and $K_t$ with `steady_state`; the RMS errors depend on the random draw.

In [ ]:
mu, sigma_p, sigma_m, mu0, sigma0 = 0.25, 0.3, 1.0, 0.0, 2.0
x, y = simulate_track(n_frames=24, mu=mu, sigma_p=sigma_p, sigma_m=sigma_m, mu0=mu0, sigma0=sigma0, seed=3)
est = kalman_filter(y, mu, sigma_p, sigma_m, mu0, sigma0)
sigma_t, K_t = np.sqrt(est["var"]), est["gain"]
sigma_inf, K_inf = steady_state(sigma_p, sigma_m)

print(f"RMS error of the observations y_t:              {rms(y, x):.2f}")
print(f"RMS error of the estimate mu_t from frame 4 on: {rms(est['mean'][3:], x[3:]):.2f}")
print(f"sigma_t for t = 1, 2, 3, 4: {fmt(sigma_t[:4])}   ...   t = 24: {sigma_t[-1]:.2f}")
print(f"K_t     for t = 1, 2, 3, 4: {fmt(K_t[:4])}   ...   t = 24: {K_t[-1]:.2f}")
print(f"fixed point of the variance recursion: sigma^2 = {sigma_inf**2:.3f}, sigma = {sigma_inf:.2f}, K = {K_inf:.2f}")

plot_track(x, y, est, f"tracking with mu = {mu}, sigma_p = {sigma_p}, sigma_m = {sigma_m}, prior N({mu0:g}, {sigma0:g}^2)")
plot_sigma_and_gain(est, sigma_inf, K_inf)

### Question 9.1 (3 points)

- Run the filter on the example: report the RMS errors, the first four values of $\sigma_t$ and its settled value, and compare the settled $\sigma_t$ and $K_t$ with `steady_state`. **(1 point)**
- Repeat with $\sigma_m \in \{0.3, 1, 3\}$ ($\sigma_p = 0.3$) and with $\sigma_p \in \{0.1, 0.6, 1\}$ ($\sigma_m = 1$): show the six tracks and their fixed-point $K$ and $\sigma$. By the weighted-average form, which source gets the weight when $\sigma_m$ grows, and when $\sigma_p$ grows? **(1 point)**
- Remove the observations of frames 10–14 (`observed`) and show the figure with the band $\mu_t \pm \sigma_t$ and the prediction (`show_prediction=True`). Why does the band widen during the gap and narrow once the observations return? **(1 point)**

In [ ]:
# Your code for Question 9.1, e.g.
# x2, y2 = simulate_track(n_frames=24, mu=mu, sigma_p=sigma_p, sigma_m=3.0, mu0=mu0, sigma0=sigma0, seed=3)
# est2 = kalman_filter(y2, mu, sigma_p, 3.0, mu0, sigma0)
# observed = np.ones(24, dtype=bool); observed[9:14] = False     # frames t = 10, ..., 14 (index 0 is t = 1)
# est_gap = kalman_filter(y, mu, sigma_p, sigma_m, mu0, sigma0, observed=observed)
# plot_track(x, y, est_gap, "observations of frames 10-14 removed", observed=observed, show_prediction=True)

**Your answers for Question 9.1:**

## Special cases: a static object and noiseless observations

Noiseless observations ($\sigma_m = 0$): Question 9.2. A static object ($\sigma_p = 0$, $\mu = 0$): the recursion computes the maximum a posteriori (MAP) estimate of the position online, reproducing exactly the batch posterior after $t$ measurements,

$$\frac{1}{\sigma_t^2} = \frac{1}{\sigma_0^2} + \frac{t}{\sigma_m^2}, \qquad \mu_t = \sigma_t^2 \Big( \frac{\mu_0}{\sigma_0^2} + \frac{1}{\sigma_m^2} \sum_{s=1}^{t} y_s \Big),$$

with $\sigma_t$ falling like $\sigma_m / \sqrt{t}$ (pp. 31–33). The cell: object at $x = 2$, 25 measurements with $\sigma_m = 1$, prior $N(0, 2^2)$ (`batch_posterior`).

In [ ]:
sigma_m, mu0, sigma0 = 1.0, 0.0, 2.0
x_static, y_static = simulate_track(n_frames=25, mu=0.0, sigma_p=0.0, sigma_m=sigma_m, mu0=mu0, sigma0=sigma0, x1=2.0, seed=3)
est_static = kalman_filter(y_static, mu=0.0, sigma_p=0.0, sigma_m=sigma_m, mu0=mu0, sigma0=sigma0)
mean_batch, var_batch = batch_posterior(y_static, sigma_m, mu0, sigma0)
sigma_static = np.sqrt(est_static["var"])
t = np.arange(1, 26)

print(f"sigma_t after 1 measurement: {sigma_static[0]:.2f}, after 25: {sigma_static[-1]:.2f}   (sigma_m / sqrt(25) = {sigma_m / 5:.2f})")
print(f"final estimate mu_25 = {est_static['mean'][-1]:.2f}; average of the 25 measurements = {y_static.mean():.2f}; true position 2")
print(f"recursion vs batch posterior: max |mu_t difference| = {np.max(np.abs(est_static['mean'] - mean_batch)):.1e}, "
      f"max |sigma_t^2 difference| = {np.max(np.abs(est_static['var'] - var_batch)):.1e}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), gridspec_kw={"width_ratios": [1.6, 1]})
plot_track(x_static, y_static, est_static, "a static object at x = 2, sigma_m = 1, prior N(0, 2^2)", ax=axes[0])
axes[1].plot(t, sigma_static, "o-", ms=3, color=COLOUR["estimate"], label=r"$\sigma_t$ (filter)")
axes[1].plot(t, np.sqrt(var_batch), "x", ms=6, color="k", label=r"$\sigma_t$ (batch posterior)")
axes[1].plot(t, sigma_m / np.sqrt(t), "--", color="0.5", label=r"$\sigma_m / \sqrt{t}$")
axes[1].set_xlabel("number of measurements $t$")
axes[1].set_ylabel(r"$\sigma_t$")
axes[1].set_title("posterior standard deviation")
axes[1].legend(fontsize=8)
fig.tight_layout()
plt.show()

### Question 9.2 (2 points)

- Report the maximum absolute difference of the means and of the variances between the recursion and the batch posterior over the 25 steps, and show the plot of $\sigma_t$ with $\sigma_m / \sqrt{t}$. Why can the MAP estimate of a static object be computed one measurement at a time? **(1 point)**
- Run `kalman_filter` on the observations `y` of the moving target with $\sigma_m = 10^{-6}$ in place of $1$ (noiseless observations). What do $\mu_t$ and the gain $K_t$ become, and why, from the update equation? **(1 point)**

In [ ]:
# Your code for Question 9.2, e.g.
# est_exact = kalman_filter(y, mu=0.25, sigma_p=0.3, sigma_m=1e-6, mu0=0.0, sigma0=2.0)

**Your answers for Question 9.2:**

## Dynamic programming on a chain

A chain of $N$ variables with $k$ values and neighbour costs $\phi_{m,m+1}(x_m, x_{m+1})$ has $k^N$ configurations; dynamic programming (DP) minimises their sum in $(N-1) k^2$ operations (pp. 61–69), computing forward, for every value of $x_m$,

$$h_m(x_m) = \min_{x_{m-1}} \{ h_{m-1}(x_{m-1}) + \phi_{m-1,m}(x_{m-1}, x_m) \},$$

storing the minimiser (the best way into $x_m$) and following the stored minimisers back from $\arg\min h_N$; DP works on any graph without closed loops (p. 72).

**The example.** A signal with $k = 5$ grey levels is observed with Gaussian noise at $N = 80$ positions; `dp_chain` finds the labelling of least energy

$$E(\mathbf{x}) = \sum_{i=1}^{N} (z_i - x_i)^2 + w \sum_{i=1}^{N-1} [x_i \ne x_{i+1}],$$

$w$ being the Potts weight of Homework 1. `chain_energy` evaluates $E$, `brute_force_chain` tries all $k^N$ configurations; the cell also builds a small instance ($N = 8$, $k = 3$).

In [ ]:
k = 5       # the signal's grey levels 0-4; Question 9.3 varies w only
truth_signal, z = make_step_signal(noise=0.5, seed=0)
N = len(z)

w = 1.0     # try other values for Question 9.3
unary, pairwise = potts_costs(z, k, w)
labels, energy_dp, h, back = dp_chain(unary, pairwise)
print(f"w = {w:g}: E(DP labelling) = {energy_dp:.2f}, E(true levels) = {chain_energy(truth_signal, unary, pairwise):.2f}, "
      f"{100 * np.mean(labels != truth_signal):.1f}% wrong labels, {np.sum(labels[1:] != labels[:-1])} changes of level "
      f"(the true signal has {np.sum(truth_signal[1:] != truth_signal[:-1])})")
plot_labellings(z, truth_signal, {f"DP labelling, w = {w:g}": labels}, k)
plot_dp_table(h, labels, f"forward table h_m(x_m) for w = {w:g}")

# A small instance with pair costs only, as in the notes: N = 8 variables, k = 3 values, integer costs 0-9 for every pair
rng = np.random.default_rng(6)
unary_small = np.zeros((8, 3))
pairwise_small = rng.integers(0, 10, size=(7, 3, 3)).astype(float)       # pairwise_small[m-1, a, b] = phi_{m,m+1}(a, b)
labels_small, cost_small, h_small, back_small = dp_chain(unary_small, pairwise_small)
for m in range(1, 8):
    print(f"h_{m + 1} = {tuple(h_small[m].astype(int).tolist())}   best way into each value: {tuple((back_small[m] + 1).tolist())}")
print(f"DP: shortest cost {cost_small:g}, path (values 1-3) {tuple((labels_small + 1).tolist())}")

### Question 9.3 (3 points)

- Run `dp_chain` on the signal for $w \in \{0, 0.5, 2, 10\}$: show the four labellings (`plot_labellings`) and report each energy of the DP labelling and of the true levels and the percentage of wrong labels. Then compare `brute_force_chain(unary_small, pairwise_small)` with `dp_chain` on the small instance. **(1 point)**
- Why does DP cost $(N-1) k^2$ operations instead of $k^N$? Give both numbers for the signal ($N = 80$, $k = 5$) and for the small instance ($N = 8$, $k = 3$). **(1 point)**
- Why can DP not be run directly on the image lattice of Homework 1, and what did we use there instead? **(1 point)**

In [ ]:
# Your code for Question 9.3, e.g.
# labellings = {}
# for w in (0, 0.5, 2, 10):
#     unary, pairwise = potts_costs(z, k, w)
#     labels, energy_dp, h, back = dp_chain(unary, pairwise)
#     ...
# labels_bf, cost_bf = brute_force_chain(unary_small, pairwise_small)

**Your answers for Question 9.3:**

## Hidden Markov models: the urn example

An HMM (pp. 76–94) has $N$ hidden states with transition probabilities $a_{ij} = P(q_{t+1} = s_j \mid q_t = s_i)$, initial probabilities $\pi_i$ and observation probabilities $b_j(m) = P(O_t = v_m \mid q_t = s_j)$; with $\lambda = (A, B, \pi)$, $P(O, Q \mid \lambda) = \pi_{q_1} b_{q_1}(O_1) \prod_{t \ge 2} a_{q_{t-1} q_t} \, b_{q_t}(O_t)$. It is the Bayes-Kalman graph with discrete states, $A$ the motion model and $B$ the observation model. The forward and backward variables $\alpha_t(i) = P(O_1, \ldots, O_t, q_t = s_i \mid \lambda)$ and $\beta_t(i) = P(O_{t+1}, \ldots, O_T \mid q_t = s_i, \lambda)$ give the state probabilities given all the observations:

$$\alpha_1(i) = \pi_i b_i(O_1), \qquad \alpha_{t+1}(j) = \Big[ \sum_{i=1}^{N} \alpha_t(i) a_{ij} \Big] b_j(O_{t+1}), \qquad P(O \mid \lambda) = \sum_{i=1}^{N} \alpha_T(i), \qquad \gamma_t(i) = P(q_t = s_i \mid O, \lambda) = \frac{\alpha_t(i) \beta_t(i)}{\sum_{j} \alpha_t(j) \beta_t(j)}.$$

**Decoding (pp. 107–114).** The greedy choice takes $\arg\max_i \gamma_t(i)$ at each $t$; Viterbi finds the whole sequence with the sum replaced by a maximum, $\delta_t(j) = \max_i \delta_{t-1}(i) a_{ij} \, b_j(O_t)$, $p^* = \max_i \delta_T(i)$, then back-tracking.

**The urns (p. 107):** three urns visited in the cycle $1, 2, 3, 1, \ldots$ (stay with probability 0.6, move on with 0.4, never back); each gives its own colour (red, blue, green) with probability 0.8, the others with 0.1; $\pi$ uniform; observations red, green, green, blue.

In [ ]:
A_urn = np.array([[0.6, 0.4, 0.0],
                  [0.0, 0.6, 0.4],
                  [0.4, 0.0, 0.6]])
B_urn = np.array([[0.8, 0.1, 0.1],      # urn 1: mostly red
                  [0.1, 0.8, 0.1],      # urn 2: mostly blue
                  [0.1, 0.1, 0.8]])     # urn 3: mostly green
pi_urn = np.array([1 / 3, 1 / 3, 1 / 3])
colour_names = ["red", "blue", "green"]
ball_colours = {0: "tab:red", 1: "tab:blue", 2: "tab:green"}
O = np.array([0, 2, 2, 1])             # red, green, green, blue

state_name = lambda path: ", ".join(f"s{i + 1}" for i in path)
alpha, p_O = forward(A_urn, B_urn, pi_urn, O)
gamma, _ = state_posteriors(A_urn, B_urn, pi_urn, O)
greedy_path = np.argmax(gamma, axis=1)
viterbi_path, p_star, delta, psi = viterbi(A_urn, B_urn, pi_urn, O)
best_bf, p_bf, total_bf = brute_force_hmm(A_urn, B_urn, pi_urn, O)

print(f"observations O = {', '.join(colour_names[o] for o in O)}")
print(f"P(O | lambda) = {p_O:.5f}  (forward algorithm)")
print("gamma_t(i):        s1    s2    s3")
for t in range(len(O)):
    print(f"           t = {t + 1}: {fmt(gamma[t])}")
print(f"greedy sequence (argmax of gamma_t at each t): {state_name(greedy_path)},  "
      f"P(Q, O | lambda) = {sequence_probability(A_urn, B_urn, pi_urn, O, greedy_path):.2e}")
print(f"Viterbi path: {state_name(viterbi_path)},  p* = P(Q*, O | lambda) = {p_star:.2e},  P(Q* | O, lambda) = {(p_star / p_O if p_O > 0 else float('nan')):.3f}")
print(f"brute force over {3 ** len(O)} sequences: best {state_name(best_bf)} with P(Q, O | lambda) = {p_bf:.2e}; "
      f"sum over all sequences = {total_bf:.5f} (equals P(O | lambda) from the forward algorithm: {np.isclose(total_bf, p_O)})")
print("Viterbi back-pointers psi_t(j) (best previous state, named from 1), rows t = 2, 3, 4:")
print(psi[1:] + 1)

plot_grid(gamma, A_urn, O, ball_colours, greedy_path=greedy_path, viterbi_path=viterbi_path,
             title="the urn HMM: gamma_t(i) as node shading, greedy choice (olive) and Viterbi path (teal)")

### Question 9.4 (3 points)

- Report $P(O \mid \lambda)$, the table of $\gamma_t(i)$, the greedy sequence with its probability, the Viterbi path with $p^*$, and the brute-force check (the best of the 81 sequences with its probability, and the sum over all of them against $P(O \mid \lambda)$). **(1 point)**
- Why can choosing the most probable state at each time separately give a sequence that the transition table $A$ forbids, while the Viterbi path is always allowed? **(1 point)**
- Point to the two lines in the loop of `forward` that are the prediction and the correction of the Bayes-Kalman filter (quote them), and say what replaces the integral $\int dx_t$ of the prediction. **(1 point)**

In [ ]:
# Your code for Question 9.4

**Your answers for Question 9.4:**

## Learning the transition probabilities

When the states are seen, $a_{ij}$ is estimated by the fraction of the transitions out of $s_i$ that go to $s_j$ (pp. 82–85; $I(\cdot)$ is the indicator):

$$\hat{a}_{ij} = \frac{\sum_{t=1}^{T-1} I(q_t = s_i, q_{t+1} = s_j)}{\sum_{t=1}^{T-1} I(q_t = s_i)}.$$

`transition_counts` does this (first cell). **Baum-Welch (pp. 116–122)** learns an HMM by EM: E-step,

$$\gamma_t(i) = P(q_t = s_i \mid O, \lambda), \qquad \zeta_t(i, j) = P(q_t = s_i, q_{t+1} = s_j \mid O, \lambda) = \frac{\alpha_t(i) \, a_{ij} \, b_j(O_{t+1}) \, \beta_{t+1}(j)}{P(O \mid \lambda)};$$

M-step,

$$\hat{\pi}_i = \gamma_1(i), \qquad \hat{a}_{ij} = \frac{\sum_{t=1}^{T-1} \zeta_t(i, j)}{\sum_{t=1}^{T-1} \gamma_t(i)}, \qquad \hat{b}_j(m) = \frac{\sum_{t=1}^{T} \gamma_t(j) \, I(O_t = v_m)}{\sum_{t=1}^{T} \gamma_t(j)},$$

summing over all the training sequences; the likelihood never decreases. `match_states` pairs the learnt states with the urns by their most likely colour; the second cell runs it on 40 sequences of 25 draws, 60 iterations from a random start (p. 122).

In [ ]:
states_short = simulate_markov_chain(A_urn, pi_urn, T=20, seed=1)
counts_short, A_hat_short = transition_counts(states_short, n_states=3)
print("observed states (named from 1):", " ".join(f"s{q + 1}" for q in states_short))
print("counts n_ij (row i = from, column j = to):")
print(counts_short.astype(int))
print("estimate A_hat = n_ij / sum_j n_ij:")
print(A_hat_short)
print(f"max |A_hat - A| = {np.max(np.abs(A_hat_short - A_urn)):.2f}")

In [ ]:
states_train, sequences = simulate_hmm(A_urn, B_urn, pi_urn, T=25, n_sequences=40, seed=0)
A_learnt, B_learnt, pi_learnt, log_lik = baum_welch(sequences, n_states=3, n_symbols=3, n_iter=60, seed=3)
perm = match_states(B_learnt)
A_matched, B_matched = A_learnt[perm][:, perm], B_learnt[perm]

print(f"log P(X | lambda): {log_lik[0]:.1f} at the random start, {log_lik[-1]:.1f} after 60 iterations; "
      f"never decreases: {bool(np.all(np.diff(log_lik) >= -1e-8))}")
print(f"most likely colour of each learnt state: {[colour_names[m] for m in np.argmax(B_learnt, axis=1)]} -> "
      f"learnt state matched to urn 1, 2, 3: {tuple((perm + 1).tolist())}")
print("learnt transition matrix A_hat (states in the order of the urns):")
print(A_matched)
print(f"max |A_hat - A| = {np.max(np.abs(A_matched - A_urn)):.2f}")
print("learnt observation probabilities B_hat (rows = urns, columns = red, blue, green):")
print(B_matched)

plt.figure(figsize=(4.8, 3))
plt.plot(log_lik, marker="o", ms=3)
plt.xlabel("iteration")
plt.ylabel(r"$\log P(X \mid \lambda)$")
plt.title("Baum-Welch on 40 sequences of 25 draws")
plt.tight_layout()
plt.show()
show_matrices([A_urn, A_matched], ["true A", r"learnt $\hat{A}$ (matched)"])

### Question 9.5 (3 points)

- Estimate $A$ by counting (`transition_counts`) from one observable sequence of length 50, 200 and 2000 drawn from `A_urn` with `seed=3`. Report $\max_{ij} |\hat{a}_{ij} - a_{ij}|$ for each length. **(1 point)**
- Run Baum-Welch on the 40 sequences of 25 draws. Plot $\log P(X \mid \lambda)$ against the iteration (it must never decrease) and report $\hat{A}$ after matching the learnt states to the urns, next to $A$. **(1 point)**
- Explain the M-step as counting with soft counts: what takes the place of the indicators $I(q_t = s_i)$ and $I(q_t = s_i, q_{t+1} = s_j)$? Why may the learnt states come out in a different order from the urns? **(1 point)**

In [ ]:
# Your code for Question 9.5, e.g.
# for T in (50, 200, 2000):
#     states = simulate_markov_chain(A_urn, pi_urn, T=T, seed=3)
#     counts, A_hat = transition_counts(states, n_states=3)
#     ...

**Your answers for Question 9.5:**